This analysis was not included in the thesis but mentioned in the appendix

In [6]:
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import mne
import pandas as pd
import hmp
from pycrostates.cluster import ModKMeans
from sklearn.preprocessing import normalize
import os

### HMP Fitting Function

In [7]:
def extract_timecourse_one_session(files, session_str, tmin, n_comp, location):
    """
    Extracts, preprocesses, and fits HMP models for a single session across multiple subjects.

    Parameters:
    - files (list of str): List of file names containing EEG data (e.g., .fif files).
    - session_str (str): String used to identify files corresponding to the session of interest.

    Returns:
    - models_dict (dict): Dictionary mapping subject IDs to their corresponding:
        - 'model': fitted HMP model object
        - 'epoch_data': preprocessed epoch-level data
        - 'init': initialization of model

    Notes:
    - Filters annotations to include only events corresponding to start and end of trial (reasoning part of experiment)
    - Applies HMP model fitting per subject.
    """
    models_dict = {}

    for file in files: 
        if session_str in file.split('_')[1][2:4]:
            print(f"Processing file: {file}")
            
            # 1. Read data
            raw = mne.io.read_raw_fif(os.path.join(epoch_data_path, file), preload=True)

           # 2. Extract annotations
            annotations = raw.annotations
            onset = annotations.onset
            description = annotations.description

            # 3. Create dataframe
            df = pd.DataFrame({"description": description, "onset": onset})

            # 4. Filter events
            df_filtered = df[df["description"].isin(["stim-all_stim", "trial_end"])].reset_index(drop=True)

            # 5. Map to event IDs
            mapping = {"stim-all_stim": 1, "trial_end": 2}
            df_filtered["event_id"] = df_filtered["description"].map(mapping)

            # 6. Convert to samples
            sfreq = int(raw.info['sfreq'])
            df_filtered["sample"] = (df_filtered["onset"] * sfreq).astype(int)

            # 7. Create previous column
            df_filtered["previous"] = 0

            # 8. Arrange columns
            df_events = df_filtered[["sample", "previous", "event_id"]]

            # 9. Convert to numpy
            events = df_events.values

            # 10. Define IDs
            event_id = {"stimulus": 1}
            resp_id = {"response": 2}

            # 11. Epoch data
            epoch_data = hmp.utils.read_mne_data(
                os.path.join(epoch_data_path, file),
                event_id=event_id,
                resp_id=resp_id,
                sfreq=500,
                events_provided=events,
                verbose=True,
                tmin=tmin,
                tmax=15
            )
            

            # 12. Transform data
            hmp_data = hmp.utils.transform_data(epoch_data, apply_standard=False, n_comp=n_comp)

            # 13. Initialize model
            model_init = hmp.models.hmp(
                data=hmp_data,
                sfreq=epoch_data.sfreq,
                cpus=4,
                event_width=50,
                distribution='gamma',
                shape=2,
                location=location
            )

            # 14. Fit model
            model = model_init.fit(verbose=False)

            # 15. Store in dictionary
            subject_id = "subject_" + file.split('_')[1][0:2]
            print(f"✅ Storing data for {subject_id}")

            models_dict[subject_id] = {
                "model": model,
                "epoch_data": epoch_data,
                "init": model_init
            }

    return models_dict

### Microstate Analysis Pipeline

In [8]:
# -------------------------------
# Concatenate trials into continuous Raw objects
# -------------------------------

def concatenate_raw_data(models_dict, info):
    """
    Concatenates trials into continuous MNE Raw objects for each subject. 
    Only EEG data relating to the reasoning phase is kept (encoding phase excluded), 
    and the inter-trial interval has been removed.
    
    Args:
        models_dict (dict): Dictionary where keys are subject IDs and values are dicts 
                            containing "epoch_data" (xarray with trial EEG data).
        info (mne.Info): MNE info object containing session metadata 

    Returns:
        dict: Dictionary mapping each subject ID to a concatenated MNE Raw object.
    """
    raw_combined_dict = {}

    for subject in models_dict:
        model_info = models_dict[subject]  
        epoch_data = model_info["epoch_data"]  

        # Concatenate trials
        raw_combined_list = []

        for trial in epoch_data.data[0].values:
            # trial is (n_channels, n_samples)
            valid_mask = ~np.isnan(trial).all(axis=0)  # remove columns that are all NaN
            trimmed = trial[:, valid_mask]

            # Convert to MNE Raw object
            raw = mne.io.RawArray(trimmed, info, verbose=False)
            raw_combined_list.append(raw)

        # Combine all Raw objects into one continuous recording
        raw_combined = mne.concatenate_raws(raw_combined_list)
        raw_combined_dict[subject] = raw_combined

    return raw_combined_dict

# -------------------------------
# Perform microstate analysis (ModKMeans clustering)
# -------------------------------

def perform_microstate_analysis(raw_combined_dict, n_clusters):
    """
    Performs microstate analysis using the modified k-means (ModKMeans) algorithm on concatenated EEG data.

    Args:
        raw_combined_dict (dict): Dictionary mapping each subject ID to a concatenated MNE Raw object.
        n_clusters (int): Number of microstate clusters to extract.

    Returns:
        dict: Dictionary mapping each subject ID to a fitted ModKMeans model.
    """
    ModK_dict = {}

    for subject in raw_combined_dict:
        raw_combined = raw_combined_dict[subject]

        # Initialize and fit model
        ModK = ModKMeans(n_clusters=n_clusters, random_state=42)
        ModK.fit(raw_combined, n_jobs=4, verbose="WARNING")

        # Store fitted model
        ModK_dict[subject] = ModK

        print(f"✅ Done with microstate analysis for subject: {subject}\n") 

    return ModK_dict

# -------------------------------
# Compute microstate segmentations
# -------------------------------

def compute_segmentation_for_ModK(ModK_dict, raw_combined_dict):
    """
    Uses ModKMeans models to compute microstate segmentations 
    for each subject's continuous EEG data.

    Args:
        ModK_dict (dict): Dictionary mapping subject IDs to fitted ModKMeans models.
        raw_combined_dict (dict): Dictionary mapping subject IDs to concatenated Raw objects.

    Returns:
        dict: Dictionary mapping each subject ID to an array of microstate labels 
              (segmentation across the continuous recording).
    """
    segmentation_dict = {}
    for subject in ModK_dict:
        ModK = ModK_dict[subject]
        raw_combined = raw_combined_dict[subject]

        segmentation = ModK.predict(
        raw_combined,
        reject_by_annotation=True,
        factor=10,
        half_window_size=10,
        min_segment_length=5,
        reject_edges=True,
        )

        segmentation_dict[subject] = segmentation

    return segmentation_dict


### Event Timing Analysis: Helper Functions for Subsequent Analysis

In [9]:
# -------------------------------
# Absolute timing (list) for one event
# -------------------------------

def event_abs_timing_list(eventprobs, event_n):
    """
    For a given event number, finds the time point in each trial where the event has the highest probability.

    Args:
        eventprobs: 3D array (trials × samples × events), probabilities for each event over time.
        event_n: Integer index of the event of interest.

    Returns:
        List of time indices (one per trial) where the given event is most probable (in sample indices).
    """
    trial_results = []

    for trial in range(len(eventprobs)):
        trial_probs = [eventprobs[trial][sample][event_n] for sample in range(len(eventprobs[trial]))]
        max_idx = trial_probs.index(max(trial_probs))
        trial_results.append(max_idx)

    return trial_results

# -------------------------------
# Absolute timing (DataFrame) for all events
# -------------------------------

def event_abs_timing_df(eventprobs, n_events):
    """
    Constructs a DataFrame showing, for each event, the sample index at which that event
    had the highest probability across trials.

    Args:
        eventprobs: 3D array (trials × samples × events), probabilities for each event over time.
        n_events: Integer, number of events.

    Returns:
        DataFrame of shape (trials × events), with each entry being the sample index of peak probability.
    """
    abs_timing_dict = {}

    for event in range(n_events):
        abs_timing_list = event_abs_timing_list(eventprobs, event)
        abs_timing_dict[event] = abs_timing_list

    return pd.DataFrame(abs_timing_dict)

# -------------------------------
# Relative timing (list) for one event
# -------------------------------

def event_rel_timing_list(eventprobs, rt, event_n):
    """
    Calculate the relative timing of a specific event within each trial,
    expressed as a proportion of the reaction time.

    Args:
        eventprobs: 3D array (trials × samples × events), probabilities for each event over time.
        rt: List of reaction times (in seconds), one per trial.
        event_n: Integer index of the event of interest.

    Returns:
        List of relative timings (one per trial).
    """
    trial_results = []

    for trial in range(len(eventprobs)):
        trial_probs = [eventprobs[trial][sample][event_n] for sample in range(len(eventprobs[trial]))]
        max_idx = trial_probs.index(max(trial_probs))

        # Convert sample index to milliseconds (500 Hz = 2 ms per sample)
        event_ms = max_idx * 2
        rt_ms = rt[trial] * 1000

        trial_results.append(event_ms / rt_ms)

    return trial_results

# -------------------------------
# Relative timing (DataFrame) for all events
# -------------------------------

def event_rel_timing_df(eventprobs, rt, n_events):
    """
    Constructs a DataFrame showing, for each event, the relative timing (as a proportion of total trial duration)
    at which that event had the highest probability across trials.

    Args:
        eventprobs: 3D array (trials × samples × events), probabilities for each event over time.
        rt: List or array of reaction times (in seconds), one per trial.
        n_events: Integer, number of events.

    Returns:
        DataFrame with shape (trials x events), with each value being the relative timing of peak probability.
    """
    rel_timing_dict = {}

    for event in range(n_events):
        rel_timing_list = event_rel_timing_list(eventprobs, rt, event)
        rel_timing_dict[event] = rel_timing_list

    return pd.DataFrame(rel_timing_dict)

### HMP versus Microstate Comparison: Similarity of Temporal Structure

In [10]:
# -------------------------------
# Find microstate transitions per subject
# -------------------------------

def get_microstate_transitions(segmentation_dict):
    """
    Finds sample indices where microstate labels change for each subject.

    Args:
        segmentation_dict (dict): Dictionary mapping subject IDs to segmentation. 
        Segmentation contains a `.labels` array, with the cluster label per sample. 

    Returns:
        dict: Dictionary mapping each subject ID to a list of transition sample indices.
    """
    transitions_dict = {}

    for subject, segmentation in segmentation_dict.items():
        labels = segmentation.labels
        df = pd.DataFrame({
            'microstate': labels,
            'sample_idx': range(len(labels))
        })

        transitions = [
            idx for idx in range(len(df) - 1)
            if df.loc[idx, 'microstate'] != df.loc[idx + 1, 'microstate']
        ]

        transitions_dict[subject] = transitions

    return transitions_dict

# -------------------------------
# Absolute event sample indices per subject
# -------------------------------

def get_HMP_transitions(absolute_dict, epoch_data_dict, sfreq=500):
    """
    For each subject, convert per-trial event onsets to absolute sample indices
    on the concatenated timeline (no inter-trial interval) by adding a cumulative 
    RT-based baseline.

    Notes:
    In `df_absolute`, each row corresponds to one trial, and the event onsets in that row
    are expressed relative to zero (trial start). This function shifts them into the
    absolute concatenated timeline by adding the cumulative trial durations (RTs).

    Args:
        absolute_dict (dict): Dictionary mapping subject IDs to DataFrames with one row 
                              per trial and columns containing event onsets (in samples).
        epoch_data_dict (dict): Dictionary mapping subject IDs to epoch objects with 
                                `.rt` array (reaction times in seconds per trial).
        sfreq (int): Sampling frequency in Hz. Default is 500.

    Returns:
        dict: Dictionary mapping each subject ID to a list of absolute sample indices 
        corresponding to transitions (from no-event to event)
    """
    results = {}
    
    for subject, absolute_df in absolute_dict.items():
        epoch_data = epoch_data_dict[subject]
    
        concatenated_list = []
        baseline = 0
    
        for idx, row in absolute_df.iterrows():
            for value in row:
                concatenated_list.append(int(round(baseline + value)))
    
            rt_samples = int(round(epoch_data.rt.values[idx] * sfreq))
            baseline += rt_samples
    
        results[subject] = concatenated_list
    
    return results


# -------------------------------
# Compare HMP and microstate transitions with tolerance
# -------------------------------

def compare_transitions(HMP_transitions, microstate_transitions, tol=20):
    """
    Compares HMP and microstate transitions within a tolerance window.

    Args:
        HMP_transitions (dict): Dictionary mapping subject IDs to lists of HMP transition sample indices.
        microstate_transitions (dict): Dictionary mapping subject IDs to lists of microstate transition sample indices.
        tol (int): Allowed difference in samples for a match. Default is 20.

    Returns:
        dict: Dictionary mapping subject IDs to results with keys:
              - "matches": number of matched transitions
              - "jaccard": Jaccard similarity score between the two sets
              - "match_rate": matches divided by number of HMP transitions
    """
    results = {}

    for subj in HMP_transitions:
        hmp = HMP_transitions[subj]
        micro = microstate_transitions[subject]

        matches = 0
        matched_micro = set()

        for h in hmp:
            # find if any micro transition is within tolerance
            close = [m for m in micro if abs(m - h) <= tol]
            if close:
                matches += 1
                matched_micro.update(close)

        denom = len(hmp) + len(micro) - matches
        jaccard = matches / denom
        match_rate = matches / len(hmp)

        results[subj] = {"matches": matches, "jaccard": jaccard, "match_rate":match_rate}

    return results

# HMP vs Microstates: Similarity in Temporal Structure

In [11]:
# -------------------------------
# Load data and path
# -------------------------------
epoch_data_path = os.path.join(os.getcwd())
files = [x for x in os.listdir(epoch_data_path) if "preprocessed-raw.fif" in x]

In [12]:
# -------------------------------
# HMP fitting
# -------------------------------
models_third_session = extract_timecourse_one_session(files, "03", -0.2, 10, 50)

Processing file: subj_0203_preprocessed-raw.fif
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0203_preprocessed-raw.fif...
    Range : 0 ... 2918399 =      0.000 ...  1425.000 secs
Ready.
Reading 0 ... 2918399  =      0.000 ...  1425.000 secs...
Processing participant C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0203_preprocessed-raw.fif's continuous eeg
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0203_preprocessed-raw.fif...
    Range : 0 ... 2918399 =      0.000 ...  1425.000 secs
Ready.
Reading 0 ... 2918399  =      0.000 ...  1425.000 secs...
Downsampling to 500 Hz
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 1.7e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 165.16 Hz
- Upper tra

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    3.7s


Creating epochs based on following event ID :[1 2]
Adding metadata with 2 columns
80 matching events found
Setting baseline interval to [-0.19921875, 0.0] s
Applying baseline correction (mode: mean)
Using data from preloaded Raw for 80 events and 31131 original time points (prior to decimation) ...
0 bad epochs dropped
Applying reaction time trim to keep RTs between 0 and 15.002 seconds
80 RTs kept of 80 clean epochs
80 trials were retained for participant C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0203_preprocessed-raw.fif
End sampling frequency is 500 Hz


  0%|          | 0/879 [00:00<?, ?it/s]

✅ Storing data for subject_02
Processing file: subj_0403_preprocessed-raw.fif
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0403_preprocessed-raw.fif...
    Range : 0 ... 2779135 =      0.000 ...  1357.000 secs
Ready.
Reading 0 ... 2779135  =      0.000 ...  1357.000 secs...
Processing participant C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0403_preprocessed-raw.fif's continuous eeg
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_0403_preprocessed-raw.fif...
    Range : 0 ... 2779135 =      0.000 ...  1357.000 secs
Ready.
Reading 0 ... 2779135  =      0.000 ...  1357.000 secs...
Downsampling to 500 Hz
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 1.7e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passba

...

✅ Storing data for subject_22
Processing file: subj_2503_preprocessed-raw.fif
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_2503_preprocessed-raw.fif...
    Range : 0 ... 1982463 =      0.000 ...   968.000 secs
Ready.
Reading 0 ... 1982463  =      0.000 ...   968.000 secs...
Processing participant C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_2503_preprocessed-raw.fif's continuous eeg
Opening raw data file C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_2503_preprocessed-raw.fif...
    Range : 0 ... 1982463 =      0.000 ...   968.000 secs
Ready.
Reading 0 ... 1982463  =      0.000 ...   968.000 secs...
Downsampling to 500 Hz
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 1.7e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passba

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.8s


Creating epochs based on following event ID :[1 2]
Adding metadata with 2 columns
80 matching events found
Setting baseline interval to [-0.19921875, 0.0] s
Applying baseline correction (mode: mean)
Using data from preloaded Raw for 80 events and 31131 original time points (prior to decimation) ...
1 bad epochs dropped
Applying reaction time trim to keep RTs between 0 and 15.002 seconds
79 RTs kept of 79 clean epochs
79 trials were retained for participant C:\Users\celia\Desktop\EEG data\Final Notebooks\subj_2503_preprocessed-raw.fif
End sampling frequency is 500 Hz


  0%|          | 0/544 [00:00<?, ?it/s]

✅ Storing data for subject_25


In [13]:
# -------------------------------
# Concatenate Data
# -------------------------------
# Create info object 
epoch_data = models_third_session["subject_02"]["epoch_data"]
ch_names = epoch_data.channels.values.tolist()
sfreq = epoch_data.sfreq
ch_types = ["eeg"]*64
info = mne.create_info(ch_names = ch_names,  sfreq = sfreq, ch_types = ch_types)
montage = mne.channels.make_standard_montage('biosemi64')
info.set_montage(montage)

# Run concatenate data function
concat_data_dict = concatenate_raw_data(models_third_session, info)

In [14]:
# -------------------------------
# Perform Microstate Analysis
# -------------------------------
ModK_dict = perform_microstate_analysis(concat_data_dict, 3)

✅ Done with microstate analysis for subject: subject_02

✅ Done with microstate analysis for subject: subject_04

✅ Done with microstate analysis for subject: subject_05

✅ Done with microstate analysis for subject: subject_06

✅ Done with microstate analysis for subject: subject_08

✅ Done with microstate analysis for subject: subject_09

✅ Done with microstate analysis for subject: subject_10

✅ Done with microstate analysis for subject: subject_11

✅ Done with microstate analysis for subject: subject_12

✅ Done with microstate analysis for subject: subject_14

✅ Done with microstate analysis for subject: subject_15

✅ Done with microstate analysis for subject: subject_16

✅ Done with microstate analysis for subject: subject_17

✅ Done with microstate analysis for subject: subject_19

✅ Done with microstate analysis for subject: subject_20

✅ Done with microstate analysis for subject: subject_22

✅ Done with microstate analysis for subject: subject_25



In [15]:
# -------------------------------
# Compute microstate segmentations
# -------------------------------
segmentation_dict = compute_segmentation_for_ModK(ModK_dict, concat_data_dict)

In [16]:
# -------------------------------
# Get sample indices for microstate transitions
# -------------------------------
microstate_transitions_dict = get_microstate_transitions(segmentation_dict)

In [17]:
# -------------------------------
# Get sample indices for HMP transitions
# -------------------------------
absolute_dict = {}
epoch_data_dict = {}

for subject in models_third_session: 
    model = models_third_session[subject]["model"]
    eventprobs = model.eventprobs.values
    n_events = eventprobs.shape[2]
    absolute_df = event_abs_timing_df(eventprobs, n_events)
    absolute_dict[subject] = absolute_df

    epoch_data = models_third_session[subject]["epoch_data"]
    epoch_data_dict[subject] = epoch_data
    
HMP_transitions_dict = get_HMP_transitions(absolute_dict, epoch_data_dict, sfreq=500)

In [27]:
# -------------------------------
# Compute similarity of temporal stucture
# -------------------------------
temp_similarity_dict = compare_transitions(HMP_transitions_dict, microstate_transitions_dict, tol=20)

In [34]:
for subject, metrics in temp_similarity_dict.items():
    match_rate = metrics["match_rate"]
    jaccard = metrics["jaccard"]
    print(f"📊 {subject}: Match Rate = {match_rate:.2f}, Jaccard = {jaccard:.2f}")

📊 subject_02: Match Rate = 0.34, Jaccard = 0.07
📊 subject_04: Match Rate = 0.32, Jaccard = 0.13
📊 subject_05: Match Rate = 0.12, Jaccard = 0.06
📊 subject_06: Match Rate = 0.15, Jaccard = 0.07
📊 subject_08: Match Rate = 0.35, Jaccard = 0.07
📊 subject_09: Match Rate = 0.32, Jaccard = 0.06
📊 subject_10: Match Rate = 0.27, Jaccard = 0.07
📊 subject_11: Match Rate = 0.23, Jaccard = 0.07
📊 subject_12: Match Rate = 0.18, Jaccard = 0.08
📊 subject_14: Match Rate = 0.17, Jaccard = 0.07
📊 subject_15: Match Rate = 0.21, Jaccard = 0.04
📊 subject_16: Match Rate = 0.19, Jaccard = 0.10
📊 subject_17: Match Rate = 0.20, Jaccard = 0.10
📊 subject_19: Match Rate = 0.15, Jaccard = 0.07
📊 subject_20: Match Rate = 0.19, Jaccard = 0.08
📊 subject_22: Match Rate = 0.17, Jaccard = 0.08
📊 subject_25: Match Rate = 0.33, Jaccard = 0.06


In [41]:
match_rate_list = []
for subject in temp_similarity_dict:
    match_rate = temp_similarity_dict[subject]["match_rate"]
    match_rate_list.append(match_rate)

print(np.mean(match_rate_list))

jaccard_sim_list = []
for subject in temp_similarity_dict:
    jaccard_sim = temp_similarity_dict[subject]["jaccard"]
    jaccard_sim_list.append(jaccard_sim)

print(np.mean(jaccard_sim_list))

0.2278490098571074
0.0749908262167746
